In [ ]:
!pip install unsloth -q
!pip install --no-deps trl peft accelerate bitsandbytes -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.

In [ ]:
import json
import random

# Generate 100 synthetic SAR training examples
training_data = []

statuses = ["prone and motionless", "waving arms", "stationary", "partially obscured by debris",
            "lying face down", "sitting upright", "crawling slowly", "standing but not moving",
            "obscured by vegetation", "waving clothing"]

hazards = ["flooding in western sectors", "structural collapse risk in grid B4-B6",
           "dense smoke reducing visibility", "unstable terrain near riverbank",
           "downed power lines in sector C", "no visible hazards", "fire approaching from north",
           "rising water levels", "debris field across sectors D-E", "none detected"]

actions = ["dispatch ground team to highest priority target",
           "maintain altitude and continue grid scan",
           "redirect to unsearched eastern sectors",
           "lower altitude for better assessment of critical targets",
           "circle back to re-verify prone subjects",
           "expand search radius to adjacent grid zones",
           "hold position and await ground team arrival"]

for i in range(100):
    num_detections = random.randint(1, 15)
    coverage = random.randint(5, 35)

    # Build detection summary
    det_lines = []
    for j in range(num_detections):
        pid = random.randint(1, 200)
        conf = round(random.uniform(0.25, 0.95), 2)
        grid_col = random.choice(['A','B','C','D','E'])
        grid_row = random.randint(1, 8)
        det_lines.append(f"- Person ID {pid}, grid {grid_col}{grid_row}, confidence {conf}")

    det_summary = "\n".join(det_lines)

    prompt = f"""You are an aerial search and rescue analyst. Be concise.

Current detections:
{det_summary}
Coverage: {coverage}/40 grid cells searched.

Generate a brief situation report."""

    # Generate ideal response
    critical_count = random.randint(0, min(3, num_detections))
    top_ids = [det_lines[k].split("ID ")[1].split(",")[0] for k in range(min(3, num_detections))]
    top_statuses = random.sample(statuses, min(3, num_detections))

    priority_lines = []
    for k in range(min(3, num_detections)):
        priority_lines.append(f"P{k+1}: #{top_ids[k]} — {top_statuses[k]} — deploy nearest unit")

    response = f"""SITUATION: {num_detections} persons detected across {coverage}/40 sectors. {critical_count} require immediate attention.
PRIORITY TARGETS:
{chr(10).join(priority_lines)}
HAZARDS: {random.choice(hazards)}
NEXT ACTION: {random.choice(actions)}"""

    training_data.append({
        "instruction": prompt,
        "output": response
    })

# Save
with open('/content/sar_training_data.json', 'w') as f:
    json.dump(training_data, f, indent=2)

print(f"Generated {len(training_data)} training examples")
print(f"\nSample prompt:\n{training_data[0]['instruction'][:200]}...")
print(f"\nSample response:\n{training_data[0]['output']}")

Generated 100 training examples

Sample prompt:
You are an aerial search and rescue analyst. Be concise.

Current detections:
- Person ID 188, grid C5, confidence 0.41
- Person ID 26, grid E5, confidence 0.81
- Person ID 30, grid C6, confidence 0.3...

Sample response:
SITUATION: 12 persons detected across 25/40 sectors. 1 require immediate attention.
PRIORITY TARGETS:
P1: #188 — obscured by vegetation — deploy nearest unit
P2: #26 — lying face down — deploy nearest unit
P3: #30 — waving arms — deploy nearest unit
HAZARDS: structural collapse risk in grid B4-B6
NEXT ACTION: expand search radius to adjacent grid zones


In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

# Load Gemma 4B with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-4b-it-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

# Format training data
import json
with open('/content/sar_training_data.json') as f:
    raw_data = json.load(f)

def format_prompt(example):
    return {
        "text": f"""<start_of_turn>user
{example['instruction']}<end_of_turn>
<start_of_turn>model
{example['output']}<end_of_turn>"""
    }

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_prompt)

print(f"Training on {len(dataset)} examples")
print(f"Sample:\n{dataset[0]['text'][:300]}...")

# Train
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        output_dir="/content/sar_finetune",
        seed=42,
    ),
)

print("Starting training...")
trainer.train()
print("Training complete!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/3.23G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

...

Step,Training Loss
10,2.968094
20,1.004336
30,0.655303
40,0.595267
50,0.555576
60,0.543841


Unsloth: Restored added_tokens_decoder metadata in /content/sar_finetune/checkpoint-60/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/sar_finetune/checkpoint-60.


Training complete!


In [ ]:
import os

# Save LoRA adapters
save_path = "/content/drive/MyDrive/ARIA/models/gemma-sar-lora"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Saved to {save_path}")

# Test the model
FastLanguageModel.for_inference(model)

test_prompt = """<start_of_turn>user
You are an aerial search and rescue analyst. Be concise.

Current detections:
- Person ID 42, grid C3, confidence 0.89
- Person ID 17, grid D5, confidence 0.72
- Person ID 8, grid B7, confidence 0.45
Coverage: 12/40 grid cells searched.

Generate a brief situation report.<end_of_turn>
<start_of_turn>model
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Test output:")
print(response.split("model\n")[-1])

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/ARIA/models/gemma-sar-lora/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/ARIA/models/gemma-sar-lora.


Saved to /content/drive/MyDrive/ARIA/models/gemma-sar-lora
Test output:
SITUATION: 3 persons detected across 12/40 sectors. 2 require immediate attention.
PRIORITY TARGETS:
P1: #42 — stationary — deploy nearest unit
P2: #17 — partially obscured by debris — deploy nearest unit
P3: #8 — crawling slowly — deploy nearest unit
HAZARDS: rising water levels
NEXT ACTION: hold position and await ground team arrival


In [ ]:
!pip install unsloth -q

from google.colab import drive
drive.mount('/content/drive')

import os
# Check if LoRA was saved
lora_path = "/content/drive/MyDrive/ARIA/models/gemma-sar-lora"
if os.path.exists(lora_path):
    print(f"Found LoRA: {os.listdir(lora_path)}")
else:
    print("LoRA not found on Drive")
    # Search for it
    for root, dirs, files in os.walk('/content/drive/MyDrive/ARIA/models'):
        print(f"{root}: {files[:5]}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 116.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22

one cell i want to gts


In [ ]:
# ── CELL 1: FULL UNSLOTH PIPELINE — RUN AND SLEEP ──

# Install
!pip install unsloth -q
!pip install --no-deps trl peft accelerate bitsandbytes -q

from google.colab import drive
drive.mount('/content/drive')

# ── Generate training data ──
import json
import random
import os

training_data = []
statuses = ["prone and motionless", "waving arms", "stationary", "partially obscured by debris",
            "lying face down", "sitting upright", "crawling slowly", "standing but not moving",
            "obscured by vegetation", "waving clothing"]
hazards = ["flooding in western sectors", "structural collapse risk in grid B4-B6",
           "dense smoke reducing visibility", "unstable terrain near riverbank",
           "downed power lines in sector C", "no visible hazards", "fire approaching from north",
           "rising water levels", "debris field across sectors D-E", "none detected"]
actions = ["dispatch ground team to highest priority target",
           "maintain altitude and continue grid scan",
           "redirect to unsearched eastern sectors",
           "lower altitude for better assessment of critical targets",
           "circle back to re-verify prone subjects",
           "expand search radius to adjacent grid zones",
           "hold position and await ground team arrival"]

for i in range(100):
    num_detections = random.randint(1, 15)
    coverage = random.randint(5, 35)
    det_lines = []
    for j in range(num_detections):
        pid = random.randint(1, 200)
        conf = round(random.uniform(0.25, 0.95), 2)
        grid_col = random.choice(['A','B','C','D','E'])
        grid_row = random.randint(1, 8)
        det_lines.append(f"- Person ID {pid}, grid {grid_col}{grid_row}, confidence {conf}")
    det_summary = "\n".join(det_lines)

    prompt = f"""You are an aerial search and rescue analyst. Be concise.

Current detections:
{det_summary}
Coverage: {coverage}/40 grid cells searched.

Generate a brief situation report."""

    critical_count = random.randint(0, min(3, num_detections))
    top_ids = [det_lines[k].split("ID ")[1].split(",")[0] for k in range(min(3, num_detections))]
    top_statuses = random.sample(statuses, min(3, num_detections))
    priority_lines = []
    for k in range(min(3, num_detections)):
        priority_lines.append(f"P{k+1}: #{top_ids[k]} — {top_statuses[k]} — deploy nearest unit")

    response = f"""SITUATION: {num_detections} persons detected across {coverage}/40 sectors. {critical_count} require immediate attention.
PRIORITY TARGETS:
{chr(10).join(priority_lines)}
HAZARDS: {random.choice(hazards)}
NEXT ACTION: {random.choice(actions)}"""

    training_data.append({"instruction": prompt, "output": response})

with open('/content/sar_training_data.json', 'w') as f:
    json.dump(training_data, f, indent=2)
print(f"Generated {len(training_data)} training examples")

# ── Load model ──
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-4b-it-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

# ── Format data ──
from datasets import Dataset

with open('/content/sar_training_data.json') as f:
    raw_data = json.load(f)

def format_prompt(example):
    return {
        "text": f"""<start_of_turn>user
{example['instruction']}<end_of_turn>
<start_of_turn>model
{example['output']}<end_of_turn>"""
    }

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_prompt)
print(f"Training on {len(dataset)} examples")

# ── Train ──
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        output_dir="/content/sar_finetune",
        seed=42,
    ),
)

print("Starting training...")
trainer.train()
print("Training complete!")

# ── Save LoRA to Drive IMMEDIATELY ──
save_path = "/content/drive/MyDrive/ARIA/models/gemma-sar-lora"
os.makedirs(save_path, exist_ok=True)
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"LoRA saved to {save_path}")

# ── Save GGUF to Drive for Ollama ──
print("Converting to GGUF (this takes a few minutes)...")
gguf_path = "/content/drive/MyDrive/ARIA/models/gemma-sar-gguf"
os.makedirs(gguf_path, exist_ok=True)
model.save_pretrained_gguf(
    gguf_path,
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"GGUF saved to {gguf_path}")

# ── Test ──
FastLanguageModel.for_inference(model)
test_prompt = """<start_of_turn>user
You are an aerial search and rescue analyst. Be concise.

Current detections:
- Person ID 42, grid C3, confidence 0.89
- Person ID 17, grid D5, confidence 0.72
- Person ID 8, grid B7, confidence 0.45
Coverage: 12/40 grid cells searched.

Generate a brief situation report.<end_of_turn>
<start_of_turn>model
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n=== TEST OUTPUT ===")
print(response.split("model\n")[-1])

# ── Verify saves ──
print("\n=== FILES SAVED ===")
for path in [save_path, gguf_path]:
    if os.path.exists(path):
        files = os.listdir(path)
        total_size = sum(os.path.getsize(os.path.join(path, f)) for f in files) / 1024 / 1024
        print(f"{path}: {len(files)} files, {total_size:.0f} MB")

print("\n✅ DONE — Safe to close. Everything saved to Drive.")

Mounted at /content/drive
Generated 100 training examples
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/3.23G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [00:44<00:44, 44.04s/it]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [01:22<00:00, 41.48s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:47<00:00, 113.83s/it]


RuntimeError: Failed to save/merge model: Unsloth: Saving LoRA finetune failed since # of LoRAs = 319 does not match # of saved modules = 0. Please file a bug report!

In [ ]:
!pip install unsloth -q
!pip install --no-deps trl peft accelerate bitsandbytes -q

from google.colab import drive
drive.mount('/content/drive')

from unsloth import FastLanguageModel
import os

# Reload the LoRA from Drive
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/drive/MyDrive/ARIA/models/gemma-sar-lora",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
print("LoRA loaded from Drive")

# Try merged save first
try:
    model.save_pretrained_merged(
        "/content/gemma-sar-merged",
        tokenizer,
        save_method="merged_16bit",
    )
    print("Merged model saved")

    # Convert merged model to GGUF
    model2, tokenizer2 = FastLanguageModel.from_pretrained(
        model_name="/content/gemma-sar-merged",
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=False,
    )
    model2.save_pretrained_gguf(
        "/content/drive/MyDrive/ARIA/models/gemma-sar-gguf",
        tokenizer2,
        quantization_method="q4_k_m",
    )
    print("GGUF saved to Drive!")
except Exception as e:
    print(f"GGUF conversion failed: {e}")
    print("LoRA is still saved — we'll use Ollama Modelfile approach instead")

# Test either way
FastLanguageModel.for_inference(model)
test_prompt = """<start_of_turn>user
You are an aerial search and rescue analyst. Be concise.

Current detections:
- Person ID 42, grid C3, confidence 0.89
- Person ID 17, grid D5, confidence 0.72
Coverage: 12/40 grid cells searched.

Generate a brief situation report.<end_of_turn>
<start_of_turn>model
"""
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7)
print("\n=== TEST ===")
print(tokenizer.decode(outputs[0], skip_special_tokens=True).split("model\n")[-1])

print("\n=== FILES ON DRIVE ===")
for folder in ['gemma-sar-lora', 'gemma-sar-gguf']:
    path = f"/content/drive/MyDrive/ARIA/models/{folder}"
    if os.path.exists(path):
        files = os.listdir(path)
        size = sum(os.path.getsize(os.path.join(path,f)) for f in files) / 1024/1024
        print(f"{folder}: {len(files)} files, {size:.0f} MB")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225

model.safetensors:   0%|          | 0.00/3.23G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [01:30<01:30, 90.90s/it]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:25<00:00, 72.83s/it]


GGUF conversion failed: Unsloth: Saving LoRA finetune failed since # of LoRAs = 319 does not match # of saved modules = 0. Please file a bug report!
LoRA is still saved — we'll use Ollama Modelfile approach instead

=== TEST ===
SITUATION: 2 persons detected across 12/40 sectors. 0 require immediate attention.
PRIORITY TARGETS: #42 — prone and stationary — deploy nearest unit
HAZARDS: fire approaching from east
NEXT ACTION: lower altitude for better assessment of highest priority target

=== FILES ON DRIVE ===
gemma-sar-lora: 8 files, 163 MB
gemma-sar-gguf: 10 files, 8239 MB


In [ ]:
import os

# Check merged model exists
merged_path = "/content/gemma-sar-merged"
if os.path.exists(merged_path):
    files = os.listdir(merged_path)
    size = sum(os.path.getsize(os.path.join(merged_path,f)) for f in files) / 1024/1024
    print(f"Merged model: {len(files)} files, {size:.0f} MB")
else:
    print("Merged model not found — need to re-merge")

Merged model: 10 files, 8239 MB


In [ ]:
!pip install unsloth -q
!pip install --no-deps trl peft accelerate bitsandbytes -q

from google.colab import drive
drive.mount('/content/drive')

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/drive/MyDrive/ARIA/models/gemma-sar-lora",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
print("LoRA loaded")

model.save_pretrained_merged(
    "/content/gemma-sar-merged",
    tokenizer,
    save_method="merged_16bit",
)
print("Merged — ready for Cell 2")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 474.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 74.2 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [ ]:
# Clone llama.cpp but DON'T install its requirements
!git clone https://github.com/ggerganov/llama.cpp.git /content/llama.cpp
!pip install gguf sentencepiece -q

import os

!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/gemma-sar-merged \
    --outfile /content/gemma-sar.gguf \
    --outtype q4_k_m

size = os.path.getsize("/content/gemma-sar.gguf") / 1024/1024
print(f"GGUF: {size:.0f} MB")

!cp /content/gemma-sar.gguf /content/drive/MyDrive/ARIA/models/gemma-sar.gguf
print("✅ DONE — gemma-sar.gguf saved to Drive")